In [1]:
import pandas as pd
import numpy as np

In [14]:
data = pd.DataFrame({
   ....:     'x0': [1, 2, 3, 4, 5],
   ....:     'x1': [0.01, -0.01, 0.25, -4.1, 0.],
   ....:     'y': [-1.5, 0., 3.6, 1.3, -2.]})

In [25]:
data.columns

Index(['x0', 'x1', 'y'], dtype='object')

In [26]:
data.to_numpy()

array([[ 1.  ,  0.01, -1.5 ],
       [ 2.  , -0.01,  0.  ],
       [ 3.  ,  0.25,  3.6 ],
       [ 4.  , -4.1 ,  1.3 ],
       [ 5.  ,  0.  , -2.  ]])

In [27]:
df2 = pd.DataFrame(data.to_numpy(), columns=['one', 'two', 'three'])   

In [28]:
df3 = df2.copy()
df3['strings'] = ['a', 'b', 'c', 'd', 'e']

In [29]:
df3.to_numpy()

array([[1.0, 0.01, -1.5, 'a'],
       [2.0, -0.01, 0.0, 'b'],
       [3.0, 0.25, 3.6, 'c'],
       [4.0, -4.1, 1.3, 'd'],
       [5.0, 0.0, -2.0, 'e']], dtype=object)

In [30]:
model_cols = ['x0', 'x1']

In [31]:
data.loc[:, model_cols].to_numpy()

array([[ 1.  ,  0.01],
       [ 2.  , -0.01],
       [ 3.  ,  0.25],
       [ 4.  , -4.1 ],
       [ 5.  ,  0.  ]])

In [32]:
data['category'] = pd.Categorical(['a', 'b', 'a', 'a', 'b'], categories=['a', 'b'])

In [33]:
data

,x0,x1,y,category
0,1,0.01,-1.5,a
1,2,-0.01,0.0,b
2,3,0.25,3.6,a
3,4,-4.10,1.3,a
4,5,0.00,-2.0,b


In [34]:
dummies = pd.get_dummies(data['category'], prefix='category')

In [36]:
data_with_dummies = data.drop('category', axis=1).join(dummies)

In [38]:
data_with_dummies

,x0,x1,y,category_a,category_b
0,1,0.01,-1.5,True,False
1,2,-0.01,0.0,False,True
2,3,0.25,3.6,True,False
3,4,-4.10,1.3,True,False
4,5,0.00,-2.0,False,True


In [39]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

In [40]:
data = pd.DataFrame({
   ....:     'x0': [1, 2, 3, 4, 5],
   ....:     'x1': [0.01, -0.01, 0.25, -4.1, 0.],
   ....:     'y': [-1.5, 0., 3.6, 1.3, -2.]})

In [41]:
data

,x0,x1,y
0,1,0.01,-1.5
1,2,-0.01,0.0
2,3,0.25,3.6
3,4,-4.10,1.3
4,5,0.00,-2.0


In [42]:
import patsy

In [43]:
y, X = patsy.dmatrices('y ~ x0 + x1', data)

In [44]:
y

DesignMatrix with shape (5, 1)
     y
  -1.5
   0.0
   3.6
   1.3
  -2.0
  Terms:
    'y' (column 0)

In [46]:
X

DesignMatrix with shape (5, 3)
  Intercept  x0     x1
          1   1   0.01
          1   2  -0.01
          1   3   0.25
          1   4  -4.10
          1   5   0.00
  Terms:
    'Intercept' (column 0)
    'x0' (column 1)
    'x1' (column 2)

In [50]:
np.asarray(y)

array([[-1.5],
       [ 0. ],
       [ 3.6],
       [ 1.3],
       [-2. ]])

In [52]:
patsy.dmatrices('y ~ x0 + x1 + 0', data)  

(DesignMatrix with shape (5, 1)
      y
   -1.5
    0.0
    3.6
    1.3
   -2.0
   Terms:
     'y' (column 0),
 DesignMatrix with shape (5, 2)
   x0     x1
    1   0.01
    2  -0.01
    3   0.25
    4  -4.10
    5   0.00
   Terms:
     'x0' (column 0)
     'x1' (column 1))

In [54]:
coef, resid, _, _ = np.linalg.lstsq(X, y, rcond=None)

In [63]:
coef = pd.Series(coef.squeeze(), index=X.design_info.column_names)

In [68]:
coef

Intercept    0.312910
x0          -0.079106
x1          -0.265464
dtype: float64

In [69]:
y, X = patsy.dmatrices('y ~ x0 + np.log(np.abs(x1) +1)', data)

In [70]:
y

DesignMatrix with shape (5, 1)
     y
  -1.5
   0.0
   3.6
   1.3
  -2.0
  Terms:
    'y' (column 0)

In [71]:
X

DesignMatrix with shape (5, 3)
  Intercept  x0  np.log(np.abs(x1) + 1)
          1   1                 0.00995
          1   2                 0.00995
          1   3                 0.22314
          1   4                 1.62924
          1   5                 0.00000
  Terms:
    'Intercept' (column 0)
    'x0' (column 1)
    'np.log(np.abs(x1) + 1)' (column 2)

In [72]:
y , X = patsy.dmatrices('y ~ standardize(x0) + center(x1)', data)

In [73]:
X

DesignMatrix with shape (5, 3)
  Intercept  standardize(x0)  center(x1)
          1         -1.41421        0.78
          1         -0.70711        0.76
          1          0.00000        1.02
          1          0.70711       -3.33
          1          1.41421        0.77
  Terms:
    'Intercept' (column 0)
    'standardize(x0)' (column 1)
    'center(x1)' (column 2)

In [ ]:
#stateful transformations
# using the original mean and std to do the transformation for the new dataset


In [82]:
new_data = pd.DataFrame({
   ....:     'x0': [6, 7, 8, 9],
   ....:     'x1': [3.1, -0.5, 0, 2.3],
   ....:     'y': [1, 2, 3, 4]})

In [83]:
new_data

,x0,x1,y
0,6,3.1,1
1,7,-0.5,2
2,8,0.0,3
3,9,2.3,4


In [84]:
new_X = patsy.build_design_matrices([X.design_info], new_data)

In [85]:
new_X

[DesignMatrix with shape (4, 3)
   Intercept  standardize(x0)  center(x1)
           1          2.12132        3.87
           1          2.82843        0.27
           1          3.53553        0.77
           1          4.24264        3.07
   Terms:
     'Intercept' (column 0)
     'standardize(x0)' (column 1)
     'center(x1)' (column 2)]

In [86]:
y, X = patsy.dmatrices('y ~ I(x0 + x1)', data)

In [87]:
X

DesignMatrix with shape (5, 2)
  Intercept  I(x0 + x1)
          1        1.01
          1        1.99
          1        3.25
          1       -0.10
          1        5.00
  Terms:
    'Intercept' (column 0)
    'I(x0 + x1)' (column 1)

In [88]:
data = pd.DataFrame({
   ....:     'key1': ['a', 'a', 'b', 'b', 'a', 'b', 'a', 'b'],
   ....:     'key2': [0, 1, 0, 1, 0, 1, 0, 0],
   ....:     'v1': [1, 2, 3, 4, 5, 6, 7, 8],
   ....:     'v2': [-1, 0, 2.5, -0.5, 4.0, -1.2, 0.2, -1.7]
   ....: })

In [89]:
data

,key1,key2,v1,v2
0,a,0,1,-1.0
1,a,1,2,0.0
2,b,0,3,2.5
3,b,1,4,-0.5
4,a,0,5,4.0
5,b,1,6,-1.2
6,a,0,7,0.2
7,b,0,8,-1.7


In [90]:
y, X = patsy.dmatrices('v2 ~ key1', data)

In [91]:
X

DesignMatrix with shape (8, 2)
  Intercept  key1[T.b]
          1          0
          1          0
          1          1
          1          1
          1          0
          1          1
          1          0
          1          1
  Terms:
    'Intercept' (column 0)
    'key1' (column 1)

In [92]:
y, X = patsy.dmatrices('v2 ~ key1 + 0', data)

In [93]:
X

DesignMatrix with shape (8, 2)
  key1[a]  key1[b]
        1        0
        1        0
        0        1
        0        1
        1        0
        0        1
        1        0
        0        1
  Terms:
    'key1' (columns 0:2)

In [94]:
data

,key1,key2,v1,v2
0,a,0,1,-1.0
1,a,1,2,0.0
2,b,0,3,2.5
3,b,1,4,-0.5
4,a,0,5,4.0
5,b,1,6,-1.2
6,a,0,7,0.2
7,b,0,8,-1.7


In [95]:
y, X = patsy.dmatrices('v2 ~ C(key2)', data)

In [96]:
X

DesignMatrix with shape (8, 2)
  Intercept  C(key2)[T.1]
          1             0
          1             1
          1             0
          1             1
          1             0
          1             1
          1             0
          1             0
  Terms:
    'Intercept' (column 0)
    'C(key2)' (column 1)

In [99]:
data['key2'] = data['key2'].map({0: 'zero', 1: 'one'})

In [100]:
data

,key1,key2,v1,v2
0,a,zero,1,-1.0
1,a,one,2,0.0
2,b,zero,3,2.5
3,b,one,4,-0.5
4,a,zero,5,4.0
5,b,one,6,-1.2
6,a,zero,7,0.2
7,b,zero,8,-1.7


In [104]:
y, X = patsy.dmatrices('v2 ~ key1 + key2 + key1:key2', data)

In [102]:
X

DesignMatrix with shape (8, 3)
  Intercept  key1[T.b]  key2[T.zero]
          1          0             1
          1          0             0
          1          1             1
          1          1             0
          1          0             1
          1          1             0
          1          0             1
          1          1             1
  Terms:
    'Intercept' (column 0)
    'key1' (column 1)
    'key2' (column 2)

In [105]:
X

DesignMatrix with shape (8, 4)
  Intercept  key1[T.b]  key2[T.zero]  key1[T.b]:key2[T.zero]
          1          0             1                       0
          1          0             0                       0
          1          1             1                       1
          1          1             0                       0
          1          0             1                       0
          1          1             0                       0
          1          0             1                       0
          1          1             1                       1
  Terms:
    'Intercept' (column 0)
    'key1' (column 1)
    'key2' (column 2)
    'key1:key2' (column 3)

In [153]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

In [154]:
rng = np.random.default_rng(seed = 12345)

In [155]:
def dnorm(mean, variance, size = 1):
    if isinstance(size, int):
        size = size,
    return mean + np.sqrt(variance) * rng.standard_normal(size)

In [156]:
N = 100
X = np.c_[
    dnorm(0,0.4,size =N),
    dnorm(0,0.6,size =N),
    dnorm(0,0.2,size =N)
]
eps = dnorm(0,0.1,size =N)
beta = [0.1, 0.3, 0.5]

y = np.dot(X, beta) + eps

In [157]:
y

array([-0.59952668, -0.58845445,  0.18563386, -0.00747657, -0.01537445,
       -0.48405182,  0.03006418,  0.21745535,  0.09733398,  0.29428061,
       -0.55821726,  0.39275937, -0.88717278, -0.14095687, -0.24884462,
       -0.11545172,  0.49031861, -0.53928623,  0.01003077, -0.12181392,
       -0.40652191, -0.26296953,  0.24121365, -0.01486882, -0.8269326 ,
        0.85796862, -0.1581605 ,  0.3229089 , -0.3182448 , -0.25177682,
        0.01201277, -0.27692688,  0.48915537,  0.02713607,  0.32623478,
       -0.67005187, -0.43638036,  0.19876113,  0.29108143,  1.22925781,
       -0.13454699,  0.11618133, -0.28334439,  0.82639714,  0.65173309,
        0.36932726,  0.46060306, -0.36001815, -0.67943794, -0.32391053,
        0.22890353,  0.33392927, -0.02893472,  0.3514912 ,  0.4104761 ,
        0.02342595, -0.08816253, -0.42223253,  0.95031167, -0.84319881,
       -0.17742481, -0.58276729, -0.04786858,  0.49981461, -0.40998529,
       -0.06505591, -0.11920059, -0.73782536,  0.11294051, -0.50

In [158]:
X_model = sm.add_constant(X)

In [159]:
X_model

array([[ 1.00000000e+00, -9.00506021e-01, -1.89429577e-01,
        -1.02787020e+00],
       [ 1.00000000e+00,  7.99252054e-01, -1.54598388e+00,
        -3.27397080e-01],
       [ 1.00000000e+00, -5.50654833e-01, -1.20254287e-01,
         3.29358994e-01],
       [ 1.00000000e+00, -1.63915546e-01,  8.24039852e-01,
         2.08274848e-01],
       [ 1.00000000e+00, -4.76512913e-02, -2.13146980e-01,
        -4.82436357e-02],
       [ 1.00000000e+00, -4.68576597e-01, -1.43558784e+00,
        -1.52694953e-01],
       [ 1.00000000e+00, -8.65068061e-01, -9.63148432e-02,
         7.08625055e-01],
       [ 1.00000000e+00,  4.10395842e-01,  6.08038650e-01,
         1.26222105e-01],
       [ 1.00000000e+00,  2.28353201e-01,  1.56467440e-01,
         4.06761512e-01],
       [ 1.00000000e+00, -1.23509905e+00, -3.31585038e-01,
         1.76681376e-01],
       [ 1.00000000e+00,  1.48463222e+00,  1.43167842e+00,
        -2.99354280e-01],
       [ 1.00000000e+00,  6.12531226e-01,  1.47169718e+00,
      

In [160]:
model = sm.OLS(y, X)

In [161]:
results = model.fit()

In [162]:
results.params

array([0.06681503, 0.26803235, 0.45052319])

In [163]:
results.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                                 OLS Regression Results                                
=======================================================================================
Dep. Variable:                      y   R-squared (uncentered):                   0.469
Model:                            OLS   Adj. R-squared (uncentered):              0.452
Method:                 Least Squares   F-statistic:                              28.51
Date:                Thu, 04 Sep 2025   Prob (F-statistic):                    2.66e-13
Time:                        14:23:28   Log-Likelihood:                         -25.611
No. Observations:                 100   AIC:                                      57.22
Df Residuals:                      97   BIC:                                      65.04
Df Model:                           3                                                  
Covariance Type:            nonrobust                                                  
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
x1             0.0668      0.054      1.243      0.217      -0.040       0.174
x2             0.2680      0.042      6.313      0.000       0.184       0.352
x3             0.4505      0.068      6.605      0.000       0.315       0.586
==============================================================================
Omnibus:                        0.435   Durbin-Watson:                   1.869
Prob(Omnibus):                  0.805   Jarque-Bera (JB):                0.301
Skew:                           0.134   Prob(JB):                        0.860
Kurtosis:                       2.995   Cond. No.                         1.64
==============================================================================

Notes:
[1] R² is computed without centering (uncentered) since the model does not contain a constant.
[2] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [164]:
data = pd.DataFrame(X, columns=['col0', 'col1', 'col2'])

In [165]:
data['y'] = y

In [166]:
data

,col0,col1,col2,y
0,-0.900506,-0.189430,-1.027870,-0.599527
1,0.799252,-1.545984,-0.327397,-0.588454
2,-0.550655,-0.120254,0.329359,0.185634
3,-0.163916,0.824040,0.208275,-0.007477
4,-0.047651,-0.213147,-0.048244,-0.015374
...,...,...,...,...
95,-0.039152,0.531515,-0.587640,-0.067934
96,-0.227355,0.941139,-0.228237,0.831554
97,-0.473484,0.167359,-0.044659,0.070316
98,-0.610622,-0.747349,-0.057917,-0.386481


In [167]:
results = smf.ols('y ~ col0 + col1 + col2', data=data).fit()

In [168]:
results.params

Intercept   -0.020799
col0         0.065813
col1         0.268970
col2         0.449419
dtype: float64

In [169]:
results.tvalues

Intercept   -0.652501
col0         1.219768
col1         6.312369
col2         6.567428
dtype: float64

In [170]:
results.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                      y   R-squared:                       0.470
Model:                            OLS   Adj. R-squared:                  0.453
Method:                 Least Squares   F-statistic:                     28.36
Date:                Thu, 04 Sep 2025   Prob (F-statistic):           3.23e-13
Time:                        14:27:10   Log-Likelihood:                -25.390
No. Observations:                 100   AIC:                             58.78
Df Residuals:                      96   BIC:                             69.20
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -0.0208      0.032     -0.653      0.516      -0.084       0.042
col0           0.0658      0.054      1.220      0.226      -0.041       0.173
col1           0.2690      0.043      6.312      0.000       0.184       0.354
col2           0.4494      0.068      6.567      0.000       0.314       0.585
==============================================================================
Omnibus:                        0.429   Durbin-Watson:                   1.878
Prob(Omnibus):                  0.807   Jarque-Bera (JB):                0.296
Skew:                           0.133   Prob(JB):                        0.863
Kurtosis:                       2.995   Cond. No.                         2.16
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [171]:
#time series processing

In [172]:
init_x = 4
values = [init_x, init_x]
N = 10000

b0 = 0.8
b1 = -0.4
noise = dnorm(0, 0.1, size=N)
for i in range(N):
    new_x = values[-1]*b0 + b1 * values[-2] + noise[i]
    values.append(new_x)


In [173]:
from statsmodels.tsa.ar_model import AutoReg


In [174]:
MAXLAGS = 5
model = AutoReg(values, lags=MAXLAGS)
results = model.fit()

In [176]:
results.params

array([ 5.29286245e-04,  7.93302666e-01, -3.87640640e-01, -5.34027682e-03,
       -5.15902880e-04, -2.80401191e-03])

In [177]:
#scikit learn

In [178]:
train = pd.read_csv('./datasets/titanic/train.csv')

In [181]:
test = pd.read_csv('./datasets/titanic/test.csv')

In [182]:
train.head(4)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S


In [183]:
train.isna().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

In [184]:
test.isna().sum()

PassengerId      0
Pclass           0
Name             0
Sex              0
Age             86
SibSp            0
Parch            0
Ticket           0
Fare             1
Cabin          327
Embarked         0
dtype: int64

In [185]:
impute_values = train['Age'].median()

In [187]:
train['Age'] = train['Age'].fillna(impute_values)

In [188]:
test['Age'] = test['Age'].fillna(impute_values)

In [189]:
train['IsFemale'] = (train['Sex'] == 'female').astype(int)
test['IsFemale'] = (test['Sex'] == 'female').astype(int)


In [192]:
predictors = ['Pclass', 'IsFemale','Age']

In [193]:
X_train = train[predictors].to_numpy()
y_train = train['Survived'].to_numpy()
X_test = test[predictors].to_numpy()

In [194]:
X_train[:5]

array([[ 3.,  0., 22.],
       [ 1.,  1., 38.],
       [ 3.,  1., 26.],
       [ 1.,  1., 35.],
       [ 3.,  0., 35.]])

In [195]:
y_train[:5]

array([0, 1, 1, 1, 0])

In [197]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression()


In [198]:
model.fit(X_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [199]:
y_predict = model.predict(X_test)

In [200]:
y_predict[:10]

array([0, 0, 0, 0, 1, 0, 1, 0, 1, 0])

In [207]:
from sklearn.linear_model import LogisticRegressionCV
model_cv = LogisticRegressionCV()

In [208]:
model_cv.fit(X_train, y_train)

,Cs,10
,fit_intercept,True
,cv,None
,dual,False
,penalty,'l2'
,scoring,None
,solver,'lbfgs'
,tol,0.0001
,max_iter,100
,class_weight,None
,n_jobs,None


In [212]:
from sklearn.model_selection import cross_val_score

In [213]:
model = LogisticRegression(C = 10)

In [214]:
scores = cross_val_score(model, X_train, y_train, cv=4)

In [215]:
scores

array([0.77578475, 0.79820628, 0.77578475, 0.78828829])